# SAV: nhận diện hành động học sinh bằng V-JEPA 2 (encoder đóng băng)

Cài đặt Kaggle: **GPU T4 x2**, **Internet On**.

Quy trình:
1. Tải annotation SAV từ Google Drive.
2. Tải video gốc qua link của tác giả, cắt các clip 3 giây giống `cut_videos.sh`, rồi xóa video gốc.
3. Encoder V-JEPA 2 ViT-L (đóng băng) chạy trên **toàn khung hình** (RoPE nên nhận được khung 16:9). Mỗi học sinh được lấy
   9 token bằng RoIAlign theo khung người. Đặc trưng được lưu ở `/kaggle/working/feats`.
4. Train head nhỏ (`linear` và `tokens`), báo frame mAP@0.5 theo giao thức AVA/SAV.

Nếu đã có clip ở một Kaggle dataset (từ phiên trước) thì đặt `CLIPS_DATASET` và bỏ qua bước 2.
Nếu đã có đặc trưng thì đặt `FEATS_DATASET` và chỉ chạy bước 4.

In [ ]:
BRANCH = "claude/elegant-lamport-ndq866"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"
MODEL = "vitl"          # vitl | vith | vitg | vitg384
QUICK_TEST = False      # True: chỉ tải 5 video và encode 20 clip mỗi split để thử pipeline
CLIPS_DATASET = ""      # vd "/kaggle/input/sav-clips/video_clips" nếu đã có clip
FEATS_DATASET = ""      # vd "/kaggle/input/sav-feats-vitl" nếu đã có đặc trưng

!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q gdown yt-dlp av timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
SAV = "/kaggle/working/sav"
CLIPS = CLIPS_DATASET or f"{SAV}/video_clips"
FEATS = FEATS_DATASET or f"/kaggle/working/feats_{MODEL}"
OUT = f"/kaggle/working/results_sav_{MODEL}"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ffmpeg -hide_banner -codecs 2>/dev/null | grep -c libx264
!cd $R && python -m pytest -q tests/test_sav.py

## 1. Annotation
In ra vài dòng đầu của file link video, để kiểm tra định dạng.

In [ ]:
!python $R/scripts/sav_download.py annotations --root $SAV
!ls -la $SAV/annotations
!head -3 $SAV/annotations/train.csv; wc -l $SAV/annotations/train.csv $SAV/annotations/val.csv $SAV/clips_to_keep.txt
!cat $SAV/annotations/education_first_label.pbtxt | head -40

## 2. Video -> clip 3 giây
Chạy được tiếp nếu bị ngắt (clip đã cắt sẽ bỏ qua). Video lỗi ghi ở `download_failures.csv`.
Nếu tỉ lệ tải được quá thấp (link chết, site chặn), dừng ở đây và báo lại.

In [ ]:
if not CLIPS_DATASET:
    LIMIT = "--limit 5" if QUICK_TEST else ""
    !python $R/scripts/sav_download.py videos --root $SAV --clips_dir $CLIPS --workers 6 --max_hours 6 $LIMIT
!ls $CLIPS | wc -l; du -sh $CLIPS
!tail -n 5 $SAV/download_failures.csv 2>/dev/null

## 3. Trích đặc trưng V-JEPA 2 (2 GPU song song)

In [ ]:
import subprocess, sys
URLS = dict(vitl="https://dl.fbaipublicfiles.com/vjepa2/vitl.pt", vith="https://dl.fbaipublicfiles.com/vjepa2/vith.pt",
            vitg="https://dl.fbaipublicfiles.com/vjepa2/vitg.pt", vitg384="https://dl.fbaipublicfiles.com/vjepa2/vitg-384.pt")
CKPT = f"/tmp/ckpt/{os.path.basename(URLS[MODEL])}"
SHORT = 384 if MODEL == "vitg384" else 256
if not FEATS_DATASET:
    !mkdir -p /tmp/ckpt && wget -q -nc -P /tmp/ckpt {URLS[MODEL]} && ls -lh /tmp/ckpt
    extra = ["--max_clips", "20"] if QUICK_TEST else []
    procs = []
    for g in range(2):
        cmd = [sys.executable, f"{R}/scripts/sav_extract.py", "--ann_dir", f"{SAV}/annotations", "--clips_dir", CLIPS,
               "--out_dir", FEATS, "--ckpt", CKPT, "--model", MODEL, "--short_side", str(SHORT),
               "--device", f"cuda:{g}", "--shard_id", str(g), "--num_shards", "2", "--max_hours", "8"] + extra
        procs.append(subprocess.Popen(cmd, stdout=open(f"/kaggle/working/extract_gpu{g}.log", "w"), stderr=subprocess.STDOUT))
    for p in procs:
        p.wait()
    print([p.returncode for p in procs])
!tail -n 6 /kaggle/working/extract_gpu0.log /kaggle/working/extract_gpu1.log
!du -sh $FEATS; cat $FEATS/missing_*.txt 2>/dev/null | wc -l

## 4. Train head + mAP
`linear`: LayerNorm + linear trên token max trong khung (giống RoI head của mmaction).
`tokens`: transformer nhỏ trên 9 token của người + 2 token ngữ cảnh của khung hình.

In [ ]:
!python $R/scripts/sav_train.py --feat_dir $FEATS --ann_dir $SAV/annotations --out_dir $OUT --head linear tokens --seeds 0 1 2
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/results.md").read()))

## Lưu lại
- `/kaggle/working/sav/video_clips`: clip đã cắt. Bấm **Save Version** rồi tạo dataset từ output để phiên sau đặt `CLIPS_DATASET`.
- `/kaggle/working/feats_*`: đặc trưng đã trích, dùng lại bằng `FEATS_DATASET` để thử head khác mà không cần GPU lâu.
- `results_sav_*/results.md`, `results.json`: gửi lại để ghi vào repo.